# Indian Amusement Parks — Data Analysis

Exploring trends in pricing, visitor volumes, revisit rates, and revenue across Indian amusement/theme/water parks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/IndianAmusementParks.csv')
print('Shape:', df.shape)
df.head(2)

In [ ]:
print('=== Data Types ===')
df.info()
print()
print('=== Summary Stats ===')
df.describe()

In [ ]:
print('=== Null Values ===')
print(df.isnull().sum())
print()
print('=== Categories ===')
print(df['category'].value_counts())
print()
print('=== Review Ratings ===')
print(df['visitor_reviews'].value_counts())

## 1. Category Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_order = df['category'].value_counts().index
sns.countplot(data=df, y='category', order=cat_order, palette='viridis', ax=axes[0])
axes[0].set_title('Number of Parks per Category')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('')

cat_pct = df['category'].value_counts(normalize=True) * 100
axes[1].pie(cat_pct.values, labels=cat_pct.index, autopct='%1.1f%%',
            startangle=90, colors=sns.color_palette('viridis', n_colors=len(cat_pct)))
axes[1].set_title('Category Share')

plt.tight_layout()
plt.show()

## 2. Price Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='category', y='price', palette='magma', ax=axes[0])
axes[0].set_title('Price Distribution by Category')
axes[0].tick_params(axis='x', rotation=45)

sns.histplot(data=df, x='price', bins=20, kde=True, color='coral', ax=axes[1])
axes[1].set_title('Overall Price Distribution')
axes[1].axvline(df['price'].mean(), color='red', ls='--', label=f"Mean: {df['price'].mean():.0f}")
axes[1].axvline(df['price'].median(), color='green', ls='--', label=f"Median: {df['price'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print('Average price by category:')
print(df.groupby('category')['price'].agg(['mean', 'median', 'min', 'max']).round(0))

## 3. Visitor Trends

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Weekly vs Monthly visitors
sns.scatterplot(data=df, x='per_week_visitors2', y='per_month_visitors',
                hue='category', size='price', sizes=(20, 200), alpha=0.7, ax=axes[0, 0])
axes[0, 0].set_title('Weekly vs Monthly Visitors')
axes[0, 0].set_xlabel('Weekly Visitors')
axes[0, 0].set_ylabel('Monthly Visitors')

# Monthly vs Yearly visitors
sns.scatterplot(data=df, x='per_month_visitors', y='per_year_visitors',
                hue='category', size='price', sizes=(20, 200), alpha=0.7, ax=axes[0, 1])
axes[0, 1].set_title('Monthly vs Yearly Visitors')
axes[0, 1].set_xlabel('Monthly Visitors')
axes[0, 1].set_ylabel('Yearly Visitors')

# Average visitors by category
avg_visitors = df.groupby('category')[['per_week_visitors2', 'per_month_visitors', 'per_year_visitors']].mean()
avg_visitors.plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Average Visitors by Category')
axes[1, 0].set_ylabel('Average Visitors')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].legend(['Weekly', 'Monthly', 'Yearly'])

# Top 10 by yearly visitors
top10_visitors = df.nlargest(10, 'per_year_visitors')[['park_name', 'category', 'per_year_visitors']]
sns.barplot(data=top10_visitors, y='park_name', x='per_year_visitors', palette='crest', ax=axes[1, 1])
axes[1, 1].set_title('Top 10 Parks by Yearly Visitors')
axes[1, 1].set_xlabel('Yearly Visitors')
axes[1, 1].set_ylabel('')

plt.tight_layout()
plt.show()

## 4. Visitor Review Ratings

In [ ]:
review_order = ['excellent', 'very good', 'good', 'average']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='visitor_reviews', order=review_order, palette='rocket', ax=axes[0])
axes[0].set_title('Distribution of Visitor Reviews')
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')

ct = pd.crosstab(df['category'], df['visitor_reviews'])
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
ct_pct.plot(kind='bar', stacked=True, color=sns.color_palette('rocket', n_colors=4), ax=axes[1])
axes[1].set_title('Review Rating Share by Category')
axes[1].set_ylabel('Percentage')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Rating', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

## 5. Revisit Rate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='category', y='revisit_rate_per_year', palette='mako', ax=axes[0])
axes[0].set_title('Revisit Rate by Category')
axes[0].set_ylabel('Revisit Rate (%)')
axes[0].tick_params(axis='x', rotation=45)

sns.scatterplot(data=df, x='price', y='revisit_rate_per_year', hue='category', alpha=0.7, ax=axes[1])
axes[1].set_title('Price vs Revisit Rate')
axes[1].set_xlabel('Ticket Price')
axes[1].set_ylabel('Revisit Rate (%)')

# Add trend line
from numpy.polynomial.polynomial import polyfit
x, y = df['price'], df['revisit_rate_per_year']
b, m = polyfit(x, y, 1)
axes[1].plot(x, b + m * x, color='red', lw=1, ls='--')

plt.tight_layout()
plt.show()

print('Average revisit rate by category:')
print(df.groupby('category')['revisit_rate_per_year'].mean().round(1).sort_values(ascending=False))

## 6. Revenue Insights

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Revenue by category
sns.boxplot(data=df, x='category', y='total_revenue_per_year(millions)', palette='YlOrRd', ax=axes[0, 0])
axes[0, 0].set_title('Revenue Distribution by Category')
axes[0, 0].set_ylabel('Revenue (Millions)')
axes[0, 0].tick_params(axis='x', rotation=45)

# Top 10 parks by revenue
top10_rev = df.nlargest(10, 'total_revenue_per_year(millions)')
sns.barplot(data=top10_rev, y='park_name', x='total_revenue_per_year(millions)',
            hue='category', dodge=False, palette='YlOrRd', ax=axes[0, 1])
axes[0, 1].set_title('Top 10 Parks by Revenue')
axes[0, 1].set_xlabel('Revenue (Millions)')
axes[0, 1].set_ylabel('')

# Revenue vs Yearly Visitors
sns.scatterplot(data=df, x='per_year_visitors', y='total_revenue_per_year(millions)',
                hue='category', size='price', sizes=(20, 200), alpha=0.7, ax=axes[1, 0])
axes[1, 0].set_title('Revenue vs Yearly Visitors')
axes[1, 0].set_xlabel('Yearly Visitors')
axes[1, 0].set_ylabel('Revenue (Millions)')

# Revenue vs Revisit Rate
sns.scatterplot(data=df, x='revisit_rate_per_year', y='total_revenue_per_year(millions)',
                hue='category', alpha=0.7, ax=axes[1, 1])
axes[1, 1].set_title('Revenue vs Revisit Rate')
axes[1, 1].set_xlabel('Revisit Rate (%)')
axes[1, 1].set_ylabel('Revenue (Millions)')

plt.tight_layout()
plt.show()

print('Average revenue by category:')
print(df.groupby('category')['total_revenue_per_year(millions)'].agg(['mean', 'sum']).round(2))

## 7. Correlation Heatmap

In [ ]:
num_cols = ['price', 'per_week_visitors2', 'per_month_visitors', 'per_year_visitors',
            'revisit_rate_per_year', 'total_revenue_per_year(millions)']

plt.figure(figsize=(9, 7))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=1)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

## 8. Key Takeaways

- **Category split**: Amusement parks dominate (~36%), followed by water parks (~24%) and theme parks (~19%). Adventure, eco, and indoor parks are niche segments.
- **Pricing**: Theme parks command the highest median prices; indoor parks are cheapest. Adventure parks show the widest price spread.
- **Visitor volumes**: Strong linear relationship between weekly, monthly, and yearly visitors — consistent footfall patterns. Water parks lead in average visitor counts.
- **Ratings**: Most parks earn 'good' or 'very good' reviews. Theme and amusement parks have the highest share of 'excellent' ratings.
- **Revisit rates**: Theme parks have the highest revisit rates (~53% avg). Indoor parks the lowest. A slight negative trend exists between price and revisit rate.
- **Revenue**: Theme parks generate the highest average revenue. Revenue strongly correlates with yearly visitor count (r ≈ 0.75), moderately with revisit rate (r ≈ 0.42).
- Overall, premium pricing alone does not drive revenue — visitor volume and repeat visits matter more.